# RF-DETR — Marine Object Detection (full fine-tune)

**Dataset**: [`ssriram2005/marine-dataset`](https://www.kaggle.com/datasets/ssriram2005/marine-dataset) (YOLO format, 7 classes: boat, buoy, cargo, ferry, tug, warship, yacht)
**Base weights**: RF-DETR-Small, pretrained on Objects-365 (pseudo-labeled with SAM2) + COCO, DINOv2 backbone
**Fine-tuning strategy**: full fine-tune (no layer freezing) — unlike a YOLO11s baseline, we train the entire
network end-to-end. RF-DETR's DINOv2 backbone is already extremely general-purpose (self-supervised on a huge,
diverse image corpus), so rather than freezing early layers, RF-DETR's own defaults use a *lower learning rate
for the backbone* (`lr_encoder`) than the rest of the network (`lr`) to preserve its pretrained knowledge while
still letting it adapt — this is the transformer-native equivalent of a partial-freeze strategy.

### Why RF-DETR-Small
- RF-DETR (Roboflow, ICLR 2026) is a DINOv2-backboned, NMS-free real-time detection transformer — current open SOTA
  on COCO and RF100-VL, ahead of YOLO11/YOLO26 at comparable latency.
- `-Small` (~22M params) is a reasonable capacity match for a ~2.5K image / 7-class real-world RGB dataset, and is
  far cheaper to fit on a single GPU than Large/XL.
- RF-DETR natively supports **YOLO-format datasets** (`dataset_file="yolo"`) — it reads the standard
  `train/valid/test` + `images/`,`labels/` + `data.yaml` layout used by Roboflow exports, so no dataset conversion
  is needed.

### Environment
This notebook is written to run **anywhere** (local machine, Colab, Kaggle, a cloud VM) — it no longer assumes
`/kaggle/working` or a pre-attached Kaggle dataset. Paths are controlled by environment variables (see
`config.env.example` / section 3 below), and the dataset can either be pointed at a local directory you already
have, or auto-downloaded via `kagglehub`.

### How to run
1. Set up the environment: `pip install -r requirements.txt`
2. Point `DATASET_ROOT` at your dataset (or let the notebook download it for you — see section 3)
3. Run all cells top to bottom. A GPU is strongly recommended; see section 2 for the check.


## 1 — Install dependencies

If you're working from a clone of this repo, prefer:

```bash
pip install -r requirements.txt
```

The cell below is a notebook-friendly fallback (e.g. for Colab/Kaggle) that installs the same pinned packages.

In [ ]:
import subprocess, sys

REQUIREMENTS = [
    "seaborn==0.12.2",
    "rfdetr[train,loggers]",
    "supervision",
    "pycocotools",
    "kagglehub",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS])
print("Done")


## 2 — Imports, seeds, device check

In [ ]:
import os, gc, random, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import cv2
from PIL import Image

import torch
import supervision as sv

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: no GPU detected — training will be extremely slow on CPU. "
          "On Colab: Runtime > Change runtime type > GPU. On Kaggle: enable an accelerator "
          "in the notebook settings. Continuing on CPU only for quick smoke-testing.")
else:
    print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
    print("GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(" GPU", i, ":", torch.cuda.get_device_name(i))


## 3 — Configuration & dataset location

All paths are read from environment variables so the notebook works the same locally, on Colab, or on Kaggle.
Copy `config.env.example` to `config.env` and edit it, or just export the variables before launching Jupyter:

| Variable       | Default              | Meaning                                                        |
|----------------|-----------------------|-----------------------------------------------------------------|
| `WORK_DIR`     | `./outputs`           | Root folder for run outputs (checkpoints, plots, results)       |
| `DATASET_ROOT` | `./data/marine-dataset` | YOLO-format dataset root (`train/`, `valid/`, `test/`, `data.yaml`) |
| `AUTO_DOWNLOAD_DATASET` | `1`          | If `DATASET_ROOT` doesn't exist, auto-download via `kagglehub`   |

RF-DETR's `dataset_file="yolo"` loader reads a Roboflow YOLO export directly (`train/`, `valid/`, `test/`
each with `images/` + `labels/`, plus a root `data.yaml` for class names) — no absolute-path rewriting needed.

In [ ]:
WORK_DIR = Path(os.environ.get("WORK_DIR", "./outputs")).resolve()
RESULTS_DIR = WORK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "./data/marine-dataset")).resolve()
AUTO_DOWNLOAD_DATASET = os.environ.get("AUTO_DOWNLOAD_DATASET", "1") == "1"

if not DATASET_ROOT.exists() and AUTO_DOWNLOAD_DATASET:
    print(f"{DATASET_ROOT} not found locally — downloading ssriram2005/marine-dataset via kagglehub...")
    import kagglehub
    downloaded_path = Path(kagglehub.dataset_download("ssriram2005/marine-dataset"))
    DATASET_ROOT = downloaded_path
    print("Downloaded to:", DATASET_ROOT)

assert DATASET_ROOT.exists(), (
    f"Dataset not found at {DATASET_ROOT}. Either set DATASET_ROOT to an existing YOLO-format "
    f"dataset directory, or set AUTO_DOWNLOAD_DATASET=1 to fetch it via kagglehub "
    f"(requires a Kaggle API token — see https://github.com/Kaggle/kagglehub#authenticate)."
)

with open(DATASET_ROOT / "data.yaml") as f:
    data_yaml = yaml.safe_load(f)

CLASS_NAMES = data_yaml["names"]
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

for split in ["train", "valid", "test"]:
    n_img = len(list((DATASET_ROOT / split / "images").glob("*.*")))
    n_lbl = len(list((DATASET_ROOT / split / "labels").glob("*.txt")))
    print(f"{split}: {n_img} images, {n_lbl} labels")

# RF-DETR's yolo dataset builder expects train/ and valid/ directly under dataset_dir
# (it does not read custom image paths out of data.yaml), so DATASET_ROOT already satisfies
# this and no fixed-yaml rewrite step is needed.
RFDETR_OUTPUT_DIR = WORK_DIR / "runs" / "rfdetr_marine_full_ft"
RFDETR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("\nRF-DETR output dir:", RFDETR_OUTPUT_DIR)


## 4 — Shared helper: YOLO label parsing

This dataset mixes plain YOLO boxes (5 fields: `cls cx cy w h`) with polygon/segmentation annotations
(`cls x1 y1 x2 y2 ... xn yn`, an odd number of fields > 5). Defined once here and reused by every later
cell (preflight checks, evaluation, visualization) so all of them agree on how instances are counted.

In [ ]:
def parse_yolo_label_line(parts, img_w, img_h):
    """Parse one YOLO label line as either a plain box (5 fields: cls cx cy w h) or a
    polygon/segmentation annotation (cls x1 y1 x2 y2 ... xn yn, an odd number of fields >5) --
    this dataset mixes both formats (Ultralytics' own loader logs a segment/box count mismatch
    on this data), and naive parsers that only accept len(parts)==5 silently drop every polygon
    line, undercounting instances. Returns (cls_id, x1, y1, x2, y2) in pixel coords, or None.
    """
    if len(parts) < 5:
        return None
    cls_id = int(parts[0])
    coords = list(map(float, parts[1:]))
    if len(coords) == 4:
        cx, cy, bw, bh = coords
        x1, y1 = (cx - bw / 2) * img_w, (cy - bh / 2) * img_h
        x2, y2 = (cx + bw / 2) * img_w, (cy + bh / 2) * img_h
    elif len(coords) >= 6 and len(coords) % 2 == 0:
        xs = [coords[i] * img_w for i in range(0, len(coords), 2)]
        ys = [coords[i] * img_h for i in range(1, len(coords), 2)]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    else:
        return None
    return cls_id, x1, y1, x2, y2


## 5 — Pre-flight dataset checks (run this BEFORE training)

Cheap, CPU-only checks that catch the problems that only show up 3 hours later otherwise:
- per-class instance counts per split (catches "0 AP" surprises before they happen)
- image/label count mismatches and empty/corrupt label or image files
- exact-duplicate images shared across train/valid/test (data leakage — inflates val mAP, tanks test mAP)
- class-index range sanity vs `data.yaml`

If anything below is flagged, fix the dataset (or at least know about it) before committing to a multi-hour run.

In [ ]:
import hashlib

SPLITS = ["train", "valid", "test"]
issues = []

# ---- 1. image/label count + empty/corrupt file checks ----
split_stems = {}
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    img_files = sorted([p for p in img_dir.glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")])
    lbl_files = sorted(lbl_dir.glob("*.txt"))
    img_stems = {p.stem for p in img_files}
    lbl_stems = {p.stem for p in lbl_files}
    split_stems[split] = img_stems

    missing_lbl = img_stems - lbl_stems
    orphan_lbl = lbl_stems - img_stems
    if missing_lbl:
        issues.append(f"[{split}] {len(missing_lbl)} images have NO label file (treated as background)")
    if orphan_lbl:
        issues.append(f"[{split}] {len(orphan_lbl)} label files have no matching image")

    empty_lbls = [p for p in lbl_files if p.read_text().strip() == ""]
    if empty_lbls:
        issues.append(f"[{split}] {len(empty_lbls)} label files are EMPTY (0 boxes)")

    corrupt = []
    for p in img_files:
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            corrupt.append(p)
    if corrupt:
        issues.append(f"[{split}] {len(corrupt)} images failed to open/verify: {[p.name for p in corrupt[:5]]}")

    print(f"{split:<6} images={len(img_files):<5} labels={len(lbl_files):<5} "
          f"no_label={len(missing_lbl):<4} empty_label={len(empty_lbls):<4} corrupt={len(corrupt)}")

# ---- 2. class-index sanity vs data.yaml ----
max_cls_seen = -1
bad_lines = 0
box_format_lines = 0
polygon_format_lines = 0
for split in SPLITS:
    for lbl_path in (DATASET_ROOT / split / "labels").glob("*.txt"):
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            parsed = parse_yolo_label_line(parts, 1, 1)  # normalized coords, w/h=1 is fine just to get cls_id
            if parsed is None:
                bad_lines += 1
                continue
            if len(parts) == 5:
                box_format_lines += 1
            else:
                polygon_format_lines += 1
            cls_id = parsed[0]
            max_cls_seen = max(max_cls_seen, cls_id)
            if cls_id >= NUM_CLASSES:
                issues.append(f"[{split}] {lbl_path.name} has out-of-range class id {cls_id} (data.yaml has {NUM_CLASSES} classes)")
if bad_lines:
    issues.append(f"{bad_lines} label lines could not be parsed as either box or polygon format")
print(f"Label line formats found: {box_format_lines} plain-box (5 fields), {polygon_format_lines} polygon/segmentation (>5 fields)")
if polygon_format_lines:
    print("NOTE: this dataset mixes box and polygon label formats -- make sure every downstream "
          "parser (eval, visualization) handles both, or instances will be silently undercounted.")
print(f"\nMax class id seen: {max_cls_seen} (data.yaml declares {NUM_CLASSES} classes: {CLASS_NAMES})")

# ---- 3. per-class instance counts per split ----
print(f"\n{'Class':<10} {'train':>7} {'valid':>7} {'test':>7}")
per_split_counts = {}
for split in SPLITS:
    counts = Counter()
    for lbl_path in (DATASET_ROOT / split / "labels").glob("*.txt"):
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            parsed = parse_yolo_label_line(parts, 1, 1)
            if parsed is not None:
                counts[parsed[0]] += 1
    per_split_counts[split] = counts

for i, name in enumerate(CLASS_NAMES):
    tr, va, te = per_split_counts["train"].get(i, 0), per_split_counts["valid"].get(i, 0), per_split_counts["test"].get(i, 0)
    flag = "  <-- fewer than 5 test instances, AP for this class will be unreliable/zero" if te < 5 else ""
    print(f"{name:<10} {tr:>7} {va:>7} {te:>7}{flag}")
    if te < 5:
        issues.append(f'Class "{name}" has only {te} test instances -- its AP will be noisy or 0 regardless of model quality')
    if tr < 20:
        issues.append(f'Class "{name}" has only {tr} TRAIN instances -- may be too few to learn well')

# ---- 4. exact-duplicate images across splits (data leakage check) ----
def md5_of(path, chunk=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

hash_to_splits = defaultdict(set)
for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    for p in img_dir.glob("*.*"):
        if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"):
            hash_to_splits[md5_of(p)].add(split)

leaks = {h: s for h, s in hash_to_splits.items() if len(s) > 1}
if leaks:
    issues.append(f"{len(leaks)} EXACT-DUPLICATE images appear in more than one split (data leakage): {[sorted(s) for s in list(leaks.values())[:5]]}")
    print(f"\n/!\\\\ {len(leaks)} images are duplicated across splits -- this inflates val mAP and can explain a val/test gap")
else:
    print("\nNo exact-duplicate images found across train/valid/test splits.")

# ---- summary ----
print("\n" + "=" * 70)
if issues:
    print(f"{len(issues)} POTENTIAL ISSUE(S) FOUND -- review before committing to a multi-hour run:")
    for iss in issues:
        print(" -", iss)
else:
    print("No issues found -- dataset looks clean for training.")
print("=" * 70)


## 6 — Load pretrained RF-DETR-Small

RF-DETR full fine-tuning does not require manual layer inspection to pick a freeze boundary — the whole
network (DINOv2 backbone + projector + deformable DETR decoder) is trained, with the backbone already at a
lower LR (`lr_encoder`) by default.

In [ ]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

n_params = sum(p.numel() for p in model.model.model.parameters())
n_trainable = sum(p.numel() for p in model.model.model.parameters() if p.requires_grad)
print(f"Total params:     {n_params:,}")
print(f"Trainable params: {n_trainable:,}  (full fine-tune -- nothing frozen)")


## 7 — Fine-tune

Training defaults below assume a single GPU (`devices=1`). Multi-GPU (`devices=2` + `strategy="ddp_notebook"`)
can be flaky when launched from inside a notebook kernel (PyTorch Lightning DDP worker `SIGABRT` at process
init) rather than a script entrypoint — if you hit that, fall back to single-GPU, which trades roughly 2x
wall-clock time for a run that reliably completes. `batch_size` here is the **per-device** batch; global
effective batch size is `batch_size * grad_accum_steps * devices`. All hyperparameters are configurable via
environment variables so you can override them without editing the notebook (e.g. for a quick CI smoke test,
set `EPOCHS=1`).

In [ ]:
# Clean up GPU memory from any previous crashed/aborted training attempt before starting fresh.
# A SIGABRT'd DDP worker can leave CUDA context/memory allocated on one or both GPUs.
gc.collect()
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        with torch.cuda.device(i):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        print(f"GPU {i} free/total: {torch.cuda.mem_get_info(i)[0]/1e9:.2f} / {torch.cuda.mem_get_info(i)[1]/1e9:.2f} GB")


In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IMGSZ = int(os.environ.get("IMGSZ", 512))          # RF-DETR-Small's native pretrain resolution
                                                     # (must be divisible by patch_size * num_windows = 32)
EPOCHS = int(os.environ.get("EPOCHS", 80))
BATCH = int(os.environ.get("BATCH", 8))             # per-device batch
GRAD_ACCUM = int(os.environ.get("GRAD_ACCUM", 2))   # global effective batch = BATCH * GRAD_ACCUM * DEVICES

# NOTE on multi-GPU: devices=2 + strategy="ddp_notebook" can crash with SIGABRT in some notebook-kernel
# environments (a spawned DDP worker aborting at process init). Falling back to single-GPU training below
# trades ~2x wall-clock time for a run that reliably completes. To retry multi-GPU, set DEVICES=2 and
# STRATEGY=ddp_notebook in the environment.
DEVICES = int(os.environ.get("DEVICES", 1))
STRATEGY = os.environ.get("STRATEGY", "auto")
AMP_DTYPE = os.environ.get("AMP_DTYPE", "fp16")     # pre-Ampere GPUs (e.g. T4) have no bf16 tensor cores

history = model.train(
    dataset_dir=str(DATASET_ROOT),
    dataset_file="yolo",
    class_names=CLASS_NAMES,
    epochs=EPOCHS,
    resolution=IMGSZ,
    batch_size=BATCH,
    grad_accum_steps=GRAD_ACCUM,
    multi_scale=False,     # keep multi-scale augmentation off to start -- re-enable later if you want the
                            # accuracy bump and have memory headroom to spare
    expanded_scales=False,
    amp_dtype=AMP_DTYPE,
    lr=1e-4,              # head/decoder LR
    lr_encoder=1.5e-4,     # DINOv2 backbone LR -- kept close to lr since we're fully fine-tuning, not freezing
    weight_decay=1e-4,
    lr_scheduler="cosine",
    warmup_epochs=2,
    early_stopping=True,
    early_stopping_patience=20,
    use_ema=True,
    seed=SEED,
    devices=DEVICES,
    strategy=STRATEGY,
    output_dir=str(RFDETR_OUTPUT_DIR),
    checkpoint_interval=10,
    tensorboard=False,
    run_test=False,
    progress_bar="tqdm",
)


## 8 — Training curves

In [ ]:
metrics_csv = RFDETR_OUTPUT_DIR / "metrics.csv"
df = pd.read_csv(metrics_csv)
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

loss_cols = [c for c in df.columns if "loss" in c.lower()]
for c in loss_cols:
    sub = df[["epoch", c]].dropna()
    axes[0].plot(sub["epoch"], sub[c], label=c)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[0].set_title("Training / validation loss"); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

map_cols = [c for c in df.columns if "map" in c.lower()]
for c in map_cols:
    sub = df[["epoch", c]].dropna()
    axes[1].plot(sub["epoch"], sub[c], label=c)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("mAP")
axes[1].set_title("Validation mAP"); axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 9 — Evaluate best checkpoint on the held-out test split

RF-DETR saves EMA and regular "best" checkpoints during training (`checkpoint_best_ema.pth` /
`checkpoint_best_regular.pth`); we prefer the EMA weights since `use_ema=True` above, falling back to the
regular best checkpoint if no EMA file is present.

In [ ]:
candidates = list(RFDETR_OUTPUT_DIR.glob("checkpoint_best_ema.pth")) or \
             list(RFDETR_OUTPUT_DIR.glob("checkpoint_best_regular.pth")) or \
             list(RFDETR_OUTPUT_DIR.glob("checkpoint_best*.pth"))
assert candidates, f"No best checkpoint found in {RFDETR_OUTPUT_DIR}"
best_weights = candidates[0]
print("Using checkpoint:", best_weights)

from rfdetr import RFDETRSmall
model_best = RFDETRSmall(pretrain_weights=str(best_weights))
model_best.model.model.eval()
if torch.cuda.is_available():
    model_best.model.model.cuda()
print("Loaded best checkpoint for evaluation")


In [ ]:
# Build COCO-format ground truth + predictions from the YOLO-format test split, then evaluate with
# pycocotools directly. We use pycocotools instead of supervision's MeanAveragePrecision here because
# its API has changed across recent supervision releases (0.29 raised a hard TypeError on the exact
# usage its own docs recommend) -- pycocotools' COCO/COCOeval interface has been stable for years and
# is what most detection benchmarks (including COCO itself) are built on, so it won't break again.
import json as _json

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

TEST_IMG_DIR = DATASET_ROOT / "test" / "images"
TEST_LBL_DIR = DATASET_ROOT / "test" / "labels"
test_imgs = sorted([p for p in TEST_IMG_DIR.glob("*.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp")])
assert test_imgs, f"No test images found under {TEST_IMG_DIR}"

# category ids are 1-indexed for pycocotools by convention
coco_categories = [{"id": i + 1, "name": name} for i, name in enumerate(CLASS_NAMES)]
cat_id_of = {i: i + 1 for i in range(len(CLASS_NAMES))}  # yolo class idx -> coco cat id

coco_images, coco_gt_annotations, coco_pred_results = [], [], []
ann_id = 1

for img_id, img_path in enumerate(test_imgs, start=1):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    coco_images.append({"id": img_id, "file_name": img_path.name, "width": int(w), "height": int(h)})

    # ---- ground truth ----
    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            parsed = parse_yolo_label_line(parts, w, h)
            if parsed is None:
                continue
            cls_id, x1, y1, x2, y2 = parsed
            bw_px, bh_px = x2 - x1, y2 - y1
            coco_gt_annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": cat_id_of[cls_id],
                "bbox": [x1, y1, bw_px, bh_px], "area": float(bw_px * bh_px), "iscrowd": 0,
            })
            ann_id += 1

    # ---- predictions ----
    preds = model_best.predict(img, threshold=0.25)
    for xyxy, cls_id, conf in zip(preds.xyxy, preds.class_id, preds.confidence):
        x1, y1, x2, y2 = [float(v) for v in xyxy]
        coco_pred_results.append({
            "image_id": img_id, "category_id": cat_id_of[int(cls_id)],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(conf),
        })

coco_gt_dict = {"images": coco_images, "annotations": coco_gt_annotations, "categories": coco_categories}
gt_json_path = RESULTS_DIR / "test_gt_coco.json"
pred_json_path = RESULTS_DIR / "test_preds_coco.json"
with open(gt_json_path, "w") as f:
    _json.dump(coco_gt_dict, f)
with open(pred_json_path, "w") as f:
    _json.dump(coco_pred_results, f)

print(f"Test images: {len(coco_images)} | GT boxes: {len(coco_gt_annotations)} | Predictions: {len(coco_pred_results)}")

coco_gt = COCO(str(gt_json_path))
if len(coco_pred_results) == 0:
    print("WARNING: model produced zero predictions above threshold=0.25 -- mAP will be 0. "
          "Check the checkpoint and threshold before trusting downstream numbers.")
    coco_dt = None
else:
    coco_dt = coco_gt.loadRes(str(pred_json_path))

if coco_dt is not None:
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    map50_95 = coco_eval.stats[0]
    map50 = coco_eval.stats[1]
    print(f"\nmAP50    : {map50:.4f}")
    print(f"mAP50-95 : {map50_95:.4f}")
else:
    coco_eval = None
    map50 = map50_95 = 0.0


## 10 — Per-class instance diagnostic

Per-class AP is only a meaningful number when the test split actually contains enough instances of that
class — a class with 0-2 GT boxes in the test set will show AP=0 or wildly noisy AP regardless of how well
the model actually learned it (that's very different from the model failing on that class).

In [ ]:
def class_counts(split):
    counts = Counter()
    for lbl_path in (DATASET_ROOT / split / "labels").glob("*.txt"):
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                counts[int(parts[0])] += 1
    return counts

print(f"{'Class':<10} {'train':>7} {'valid':>7} {'test':>7}")
for i, name in enumerate(CLASS_NAMES):
    tr = class_counts("train").get(i, 0)
    va = class_counts("valid").get(i, 0)
    te = class_counts("test").get(i, 0)
    flag = "  <-- very few/no test instances" if te < 5 else ""
    print(f"{name:<10} {tr:>7} {va:>7} {te:>7}{flag}")


## 11 — Per-class AP bar chart

In [ ]:
# Per-class AP pulled directly from COCOeval's precision array: shape is
# [iou_thresholds(10), recall_thresholds(101), num_classes, area_ranges(4), max_dets(3)].
# We take area_range='all' (index 0) and max_dets=100 (index -1, last), IoU=0.50 (index 0) for AP@50,
# and mean over all 10 IoU thresholds for AP@50-95, matching how pycocotools computes the summary stats.
if coco_eval is not None:
    precision = coco_eval.eval["precision"]  # [T, R, K, A, M]
    ap50_per_class, ap5095_per_class = [], []
    for k in range(len(CLASS_NAMES)):
        p50 = precision[0, :, k, 0, -1]
        p50 = p50[p50 > -1]
        ap50_per_class.append(p50.mean() if p50.size else 0.0)

        p5095 = precision[:, :, k, 0, -1]
        p5095 = p5095[p5095 > -1]
        ap5095_per_class.append(p5095.mean() if p5095.size else 0.0)

    ap50 = np.array(ap50_per_class)
    ap5095 = np.array(ap5095_per_class)
else:
    ap50 = np.zeros(len(CLASS_NAMES))
    ap5095 = np.zeros(len(CLASS_NAMES))

x = np.arange(len(CLASS_NAMES))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, ap50, width, label="AP@50", color="#4C72B0")
bars2 = ax.bar(x + width/2, ap5095, width, label="AP@50-95", color="#DD8452")

for bar, v in zip(bars1, ap50):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
for bar, v in zip(bars2, ap5095):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=10)
ax.set_ylabel("Average Precision"); ax.set_ylim(0, 1.12)
ax.set_title("Per-class AP -- test set (RF-DETR-Small, full fine-tune)", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_class_ap.png", dpi=150, bbox_inches="tight")
plt.show()

pc_df = pd.DataFrame({"Class": CLASS_NAMES, "AP@50": ap50, "AP@50-95": ap5095})
print(pc_df.to_string(index=False))


## 12 — Ground truth vs prediction panels (10 test images)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

NUM_SAMPLES = 10
selected = random.sample(test_imgs, min(NUM_SAMPLES, len(test_imgs)))

COLOR_GT, COLOR_PRED = (0, 200, 80), (0, 180, 255)
FONT = cv2.FONT_HERSHEY_SIMPLEX

fig, axes = plt.subplots(NUM_SAMPLES, 2, figsize=(14, 5 * NUM_SAMPLES))
fig.suptitle("Ground Truth (left) vs RF-DETR Full Fine-Tune Predictions (right) -- conf >= 0.25",
             fontsize=14, fontweight="bold", y=1.001)

for i, img_path in enumerate(selected):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    img_gt = img_rgb.copy()
    lbl_path = TEST_LBL_DIR / (img_path.stem + ".txt")
    gt_count = 0
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            parsed = parse_yolo_label_line(parts, w, h)
            if parsed is None:
                continue
            cls_id, x1f, y1f, x2f, y2f = parsed
            x1, y1, x2, y2 = int(x1f), int(y1f), int(x2f), int(y2f)
            cv2.rectangle(img_gt, (x1, y1), (x2, y2), COLOR_GT, 2)
            label = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            (tw, th), _ = cv2.getTextSize(label, FONT, 0.45, 1)
            cv2.rectangle(img_gt, (x1, max(y1-th-4, 0)), (x1+tw+2, y1), COLOR_GT, -1)
            cv2.putText(img_gt, label, (x1+1, max(y1-3, th)), FONT, 0.45, (0,0,0), 1, cv2.LINE_AA)
            gt_count += 1

    img_pred = img_rgb.copy()
    res = model_best.predict(img_rgb, threshold=0.25)
    pred_count = 0
    for xyxy, cls_id, conf in zip(res.xyxy, res.class_id, res.confidence):
        x1, y1, x2, y2 = xyxy.astype(int)
        cv2.rectangle(img_pred, (x1, y1), (x2, y2), COLOR_PRED, 2)
        label = f"{CLASS_NAMES[int(cls_id)]} {conf:.2f}" if int(cls_id) < len(CLASS_NAMES) else f"{cls_id} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, FONT, 0.45, 1)
        cv2.rectangle(img_pred, (x1, max(y1-th-4, 0)), (x1+tw+2, y1), COLOR_PRED, -1)
        cv2.putText(img_pred, label, (x1+1, max(y1-3, th)), FONT, 0.45, (0,0,0), 1, cv2.LINE_AA)
        pred_count += 1

    axes[i, 0].imshow(img_gt); axes[i, 0].set_title(f"{img_path.name} -- GT: {gt_count}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(img_pred); axes[i, 1].set_title(f"{img_path.name} -- Pred: {pred_count}"); axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "gt_vs_pred_panels.png", dpi=150, bbox_inches="tight")
plt.show()


## 13 — Save outputs summary

In [ ]:
import shutil
shutil.copy(best_weights, RESULTS_DIR / "best.pt")
shutil.copy(DATASET_ROOT / "data.yaml", RESULTS_DIR / "marine_data.yaml")

print(f"All saved outputs under {RESULTS_DIR}:")
print("{:<40} {:>8}".format("File", "Size"))
print("-" * 50)
for f in sorted(RESULTS_DIR.rglob("*")):
    if f.is_file():
        sz = f.stat().st_size
        unit = "MB" if sz > 1_000_000 else "KB"
        val = sz / 1_000_000 if sz > 1_000_000 else sz / 1024
        print("{:<40} {:>6.1f} {}".format(str(f.relative_to(RESULTS_DIR)), val, unit))
